# 🏥 Diabetes Risk Classifier Model Training (Google Colab)

This Jupyter Notebook was custom-developed for **Google Colab** to train and serialize the clinical predictive models used in the **Clini-SHAP Intelligent CDSS (Clinical Decision Support Suite)**.

### 📋 Dataset Context: Pima Indians Diabetes Database
### 🤖 Model Classifier Architecture: `XGBoost Classifier`

---

## ⚙️ Step 1: Environment Setup & Installations
We install the correct medical model dependencies including `xgboost`, `scikit-learn`, `shap`, `joblib`, `matplotlib`, and `seaborn` directly in Google Colab.


In [ ]:
# Install libraries in Google Colab
!pip install pandas numpy scikit-learn shap joblib matplotlib seaborn xgboost fpdf2


## 🧪 Step 2: Import Core Libraries
Load libraries required for numerical computation, exploratory data analysis (EDA), scaling pipeline, and SHAP interpretability.


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
import shap
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

# Setup visualization parameters
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 📂 Step 3: Load Clinical Dataset
We fetch the dataset from the live repository and load it into a pandas DataFrame.

Downloads the historical Pima Indians diagnostic metrics containing 768 patient samples directly.


In [ ]:
# Load Pima Indians Diabetes Dataset directly
dataset_url = 'https://raw.githubusercontent.com/anushka06onu/AI-Healthcare-Analytics-Platform/main/data/diabetes.csv'
try:
    df = pd.read_csv(dataset_url)
    print('Successfully downloaded diabetes dataset!')
except Exception as e:
    print('Fallback loading local data or file upload:')
    # If offline, use a dummy dataframe matching the format:
    # df = pd.read_csv('diabetes.csv')
    raise e

print(f'Shape of Diabetes Cohort: {df.shape}')
df.head()


## 🔬 Step 4: Train-Test Split and Clinical Preprocessing
We partition our EMR metrics into training and test cohorts (80% training, 20% test stratified) and scale features using `StandardScaler`.


In [ ]:
# Split features and outcomes
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# Perform stratified training split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize and fit StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print('Preprocessed baseline split succeeded!')
print(f'Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}')


## 🤖 Step 5: Train EMR Predictive Classifier
We train the high-fidelity `XGBoost Classifier` model on our preprocessed training cohort.


In [ ]:
# Train XGBClassifier model
model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_train_scaled, y_train)

# Evaluate prediction accuracy
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print('--- CLINICAL DIAGNOSTIC METRICS ---')
print(f'Accuracy Score:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision Score: {precision_score(y_test, y_pred):.4f}')
print(f'Recall Score:    {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score:        {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC Score:   {roc_auc_score(y_test, y_pred_proba):.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred))


## 🔍 Step 6: Local and Global SHAP Attributions
To ensure 100% medical interpretability and accountabilities, we initialize SHAP explainer objects and pre-calculate SHAP matrices.


In [ ]:
# Set up a KernelExplainer sample for XGBoost
print('Initializing SHAP Kernel Explainer background sample...')
bg_sample = shap.sample(X_train_scaled, min(30, len(X_train_scaled)))
test_sample = shap.sample(X_test_scaled, min(100, len(X_test_scaled)))

explainer = shap.KernelExplainer(model.predict_proba, bg_sample)
shap_values = explainer.shap_values(test_sample)

# Extract target index for probability of positive disease predictions
if isinstance(shap_values, list):
    shap_matrix = shap_values[1]
else:
    shap_matrix = shap_values

# Plot global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_matrix, test_sample, plot_type='bar', show=False)
plt.title('Diabetes Global Feature Importance (SHAP)', fontsize=14)
plt.tight_layout()
plt.show()


## 💾 Step 7: Export Serialized Components
We export the trained model classifier, the calibrated scaler, baseline training feature names, and pre-computed SHAP files for seamless integration into the CDSS application.


In [ ]:
# Ensure export directories exist
os.makedirs('models', exist_ok=True)
os.makedirs('shap_files', exist_ok=True)

# Save model & scalers
joblib.dump(model, 'models/diabetes_model.pkl')
joblib.dump(scaler, 'models/diabetes_scaler.pkl')
joblib.dump(list(scaler.feature_names_in_), 'models/diabetes_columns.joblib')
joblib.dump(X_train, 'models/diabetes_X_train.joblib')

# Save SHAP pre-computations
joblib.dump(None, 'shap_files/diabetes_explainer.joblib')
joblib.dump(shap_matrix, 'shap_files/diabetes_shap_values.joblib')
joblib.dump(test_sample, 'shap_files/diabetes_X_test.joblib')

print('Diabetes EMR serializations successfully completed!')
